# Prep

## Clone git repo

In [ ]:
!git clone -b handson https://github.com/linnabraham/galactic-rings.git

In [1]:
%cd galactic-rings

/home/linn/2024/dec/aiml-handson/galactic-rings


## Setup environment

In [5]:
%%capture
!pip install wandb
!pip install gdown

In [ ]:
%%capture
!pip install tensorflow

## Imports

In [2]:
import os
import json
import tensorflow as tf
from tensorflow.keras.callbacks import ModelCheckpoint, Callback
from alexnet_utils.params import parser, print_arguments
from alexnet_utils.alexnet import AlexNet
import wandb

2025-01-05 15:35:47.755831: I tensorflow/core/util/port.cc:110] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-01-05 15:35:50.010263: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: SSE4.1 SSE4.2 AVX AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [3]:
print(tf.__version__)

2.12.1


In [4]:
print(wandb.__version__)

0.15.11


In [5]:
print(tf.config.list_physical_devices('GPU'))

[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


## Create datasets

In [10]:
!gdown --fuzzy "https://drive.google.com/file/d/1RlPl3WD4JDz5N-kx5g00YveLtZ43vPw-/view?usp=drive_link"

Downloading...
From (original): https://drive.google.com/uc?id=1RlPl3WD4JDz5N-kx5g00YveLtZ43vPw-
From (redirected): https://drive.google.com/uc?id=1RlPl3WD4JDz5N-kx5g00YveLtZ43vPw-&confirm=t&uuid=8133badd-4b5e-429d-bf5f-08a8fe52c446
To: /home/linn/2024/dec/aiml-handson/galactic-rings/galaxies.tar.gz
100%|██████████████████████████████████████| 77.6M/77.6M [00:05<00:00, 15.1MB/s]


In [11]:
!echo "0f456e955b0b5312aec8d2dd6186218c  galaxies.tar.gz" | md5sum -c

galaxies.tar.gz: OK


In [12]:
%%capture
!tar xvzf galaxies.tar.gz -C data/

# Train

## Define argparse arguments

In [10]:
parser.add_argument('-images', '--images', required=True, help="path containing images of two classes")
parser.add_argument('-epochs', '--epochs', required=True, type=int, default=50, help="num epochs")
parser.add_argument('-model-path', '--model-path', default=None, help="Filepath to save model during training and to load model from when testing")
parser.add_argument('-val-dir', '--val-dir', default=None, help="path containing validation data")
parser.add_argument('-retrain', '--retrain', action="store_true", help="Whether to continue previous training")

_StoreTrueAction(option_strings=['-retrain', '--retrain'], dest='retrain', nargs=0, const=True, default=False, type=None, choices=None, required=False, help='Whether to continue previous training', metavar=None)

In [11]:
args = parser.parse_args(['-images', 'data/galaxies', '-epochs', '2'])

In [12]:
print_arguments(parser, args)

Arguments and Data Types:
  target_size: tuple_type - (240, 240)
  batch_size: int - 16
  train_frac: float - 0.8
  random_state: int - 42
  num_classes: int - 2
  channels: int - 3
  output_dir: None - output
  augmentation_types: str - ['flip', 'rotation']
  images: None - data/galaxies
  epochs: int - 2
  model_path: None - None
  val_dir: None - None


## Define AlexNet architecture

In [9]:
model = AlexNet.build(width=args.target_size[0], height=args.target_size[1], \
                      depth=args.channels, classes=1, reg=0.0002)

/data/linn/miniconda3/envs/aiml/lib/python3.10/site-packages/keras/src/layers/convolutional/base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [10]:
print(model.summary())

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 120, 120, 96)   │         7,296 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation (Activation)         │ (None, 120, 120, 96)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 120, 120, 96)   │           384 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 59, 59, 96)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 59, 59, 96)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 59, 59, 256)    │       614,656 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_1 (Activation)       │ (None, 59, 59, 256)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 59, 59, 256)    │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 29, 29, 256)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 29, 29, 256)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 29, 29, 384)    │       885,120 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_2 (Activation)       │ (None, 29, 29, 384)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 29, 29, 384)    │         1,536 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 29, 29, 384)    │     1,327,488 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_3 (Activation)       │ (None, 29, 29, 384)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, 29, 29, 384)    │         1,536 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_4 (Conv2D)               │ (None, 29, 29, 256)    │       884,992 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_4 (Activation)       │ (None, 29, 29, 256)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_4           │ (None, 29, 29, 256)    │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 14, 14, 256)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 14, 14, 256)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 50176)          │             0 │
├─────────────────────────────────┼────────────────────────┼─────────────

 Total params: 226,068,225 (862.38 MB)

 Trainable params: 226,049,089 (862.31 MB)

 Non-trainable params: 19,136 (74.75 KB)

None


## Define validation loss, evaluation metrics, optimizer and learning rate

In [11]:
classification_threshold = 0.5

METRICS = [
      tf.keras.metrics.Precision(thresholds=classification_threshold,
                                 name='precision'),
      tf.keras.metrics.Recall(thresholds=classification_threshold,
                              name="recall"),
      tf.keras.metrics.AUC(num_thresholds=100, curve='PR', name='auc_pr'),
]

In [12]:
model.compile(loss="binary_crossentropy", optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3)\
              , metrics=METRICS)

## Data augmentations

* Rescale to between (0,1)
* Custom augmentations
  * Rotation
  * Flip
  * Brightness
  * Contrast

In [13]:
def random_choice(x, size, seed, axis=0, unique=True):
    dim_x = tf.cast(tf.shape(x)[axis], tf.int64)
    indices = tf.range(0, dim_x, dtype=tf.int64)
    sample_index = tf.random.shuffle(indices,seed=seed)[:size]
    sample = tf.gather(x, sample_index, axis=axis)

    return sample, sample_index

def random_int_rot_img(inputs,seed):
    angles = tf.constant([1, 2, 3, 4])
    # Make a new seed.
    #new_seed = tf.random.experimental.stateless_split((seed,seed), num=1)[0, :]
    angle = random_choice(angles,1,seed=seed)[0][0]
    inputs = tf.image.rot90(inputs, k=angle)

    return inputs

def rescale(image, label):
    image = tf.cast(image, tf.float32)
    image = (image / 255.0)

    return image, label

# define custom augmentations
def augment_custom(images, labels, augmentation_types, seed):
    images, labels = rescale(images, labels)
    # Make a new seed.
    #new_seed = tf.random.experimental.stateless_split((seed,seed), num=1)[0, :]
    new_seed = seed
    if 'rotation' in augmentation_types:
        images = random_int_rot_img(images,seed=seed)
    if 'flip' in augmentation_types:
        images = tf.image.random_flip_left_right(images, seed=new_seed)
        images = tf.image.random_flip_up_down(images, seed=new_seed)
    if 'brightness' in augmentation_types:
        images = tf.image.random_brightness(images, max_delta=0.2, seed=new_seed)
    if 'contrast' in augmentation_types:
        images = tf.image.random_contrast(images, lower=0.2, upper=0.5, seed=new_seed)

    return (images, labels)

## Define Callbacks

* Model checkpoint
* Save history
* Wandb logging

In [14]:
class SaveHistoryCallback(Callback):
    def __init__(self, file_path):
        super().__init__()
        self.file_path = file_path
        self.history = {'loss': [], 'val_loss': [], 'auc_pr':[], 'val_auc_pr':[], 'val_precision':[], 'val_recall':[]}

    def on_epoch_end(self, epoch, logs=None):
        self.history['loss'].append(logs.get('loss'))
        self.history['val_loss'].append(logs.get('val_loss'))
        self.history['auc_pr'].append(logs.get('auc_pr'))
        self.history['val_auc_pr'].append(logs.get('val_auc_pr'))
        self.history['val_precision'].append(logs.get('val_precision'))
        self.history['val_recall'].append(logs.get('val_recall'))

        with open(self.file_path, 'w') as f:
            json.dump(self.history, f)

In [15]:
def create_callbacks(run_name):
    outdir = os.path.join("output", run_name)
    if not os.path.exists(outdir):
        os.makedirs(outdir)
    model_path = os.path.join(outdir,"best_model.keras")
    mc = ModelCheckpoint(model_path, monitor='val_loss', \
        mode='min', verbose=1, save_best_only=True)
    history_path = os.path.join(outdir,'history.json')
    hc = SaveHistoryCallback(history_path)
    callbacks=[mc,hc, wandb.keras.WandbMetricsLogger()]
    return callbacks

## Create tf.data.Dataset

In [16]:
def get_train_data(data_dir, val_dir, train_frac, target_size, batch_size, augmentation_types, outdir, random_state):
  if val_dir is None:
      train_ds, val_ds = tf.keras.utils.image_dataset_from_directory(
        data_dir,
        validation_split=1-train_frac,
        subset="both",
        color_mode='rgb',
        seed=random_state,
        image_size=target_size,
        batch_size=None)
  else:
      train_ds = tf.keras.utils.image_dataset_from_directory(
        data_dir,
        color_mode='rgb',
        seed=random_state,
        image_size=target_size,
        batch_size=None)

      val_ds = tf.keras.utils.image_dataset_from_directory(
            val_dir,
            color_mode='rgb',
            seed=random_state,
            image_size=target_size,
            batch_size=None)

  class_names = train_ds.class_names
  print("Training dataset class names are :",class_names)

  AUTOTUNE = tf.data.AUTOTUNE

  train_ds = (
          train_ds
          .shuffle(1000)
          .map(lambda x, y: augment_custom(x, y, augmentation_types, seed=random_state), num_parallel_calls=AUTOTUNE)
          #.cache()
          .batch(batch_size)
          .prefetch(buffer_size=AUTOTUNE)
          )

  val_ds = (
          val_ds
          .map(rescale, num_parallel_calls=AUTOTUNE)
          #.cache()
          .batch(batch_size)
          .prefetch(buffer_size=AUTOTUNE)
          )

  return train_ds, val_ds

In [17]:
train_ds, val_ds = get_train_data(args.images, args.val_dir, args.train_frac, args.target_size, args.batch_size,\
                                  args.augmentation_types, args.output_dir, args.random_state)

Found 15229 files belonging to 2 classes.
Using 12184 files for training.
Using 3045 files for validation.
Training dataset class names are : ['NonRings', 'Rings']


In [ ]:
wandb.init(project="aiml-handson", anonymous="allow")
callbacks = create_callbacks(wandb.run.name)
history = model.fit(train_ds, validation_data=val_ds, epochs=args.epochs, shuffle=True, callbacks=callbacks)
wandb.finish()

wandb: Using wandb-core as the SDK backend.  Please refer to https://wandb.me/wandb-core for more information.
wandb: Currently logged in as: linn-official. Use `wandb login --relogin` to force relogin


Epoch 1/2


# Results

[Training history comparison](https://wandb.ai/linn-official/Ring_Train/reports/val_loss-25-01-04-10-29-05---VmlldzoxMDgxMDUwMA?accessToken=aexjbxpy9q24ikedi0vf7k3edss8jszlldy15or6blpt5f3kaxxps6lt8ql3qgg5)

### Download trained model

In [31]:
!gdown --fuzzy "https://drive.google.com/file/d/1m4oVnlxAC9MxXZsQU9oEfAeFYVzmEtsA/view?usp=sharing"

Downloading...
From (original): https://drive.google.com/uc?id=1m4oVnlxAC9MxXZsQU9oEfAeFYVzmEtsA
From (redirected): https://drive.google.com/uc?id=1m4oVnlxAC9MxXZsQU9oEfAeFYVzmEtsA&confirm=t&uuid=c82cacb8-9c4f-45cb-ab1e-362fa283e7de
To: /home/linn/2024/dec/aiml-handson/galactic-rings/clean-shadow-84-slim.zip
100%|██████████████████████████████████████| 2.46G/2.46G [00:59<00:00, 40.9MB/s]


In [32]:
%%capture
!unzip clean-shadow-84-slim.zip

In [33]:
!ls clean-shadow-84-slim

best_model.h5  history.json  train_filenames.csv  validation_filenames.csv


In [34]:
!echo "6f3c1140c1b4a7f0cb02ceecaf5cb030  clean-shadow-84-slim.zip" | md5sum -c

clean-shadow-84-slim.zip: OK


### Download training data with visual selections

In [55]:
!gdown --fuzzy "https://drive.google.com/file/d/1YPWqAfXbltU56Biz7j2_nyAWlyODEdUA/view?usp=sharing"

Downloading...
From (original): https://drive.google.com/uc?id=1YPWqAfXbltU56Biz7j2_nyAWlyODEdUA
From (redirected): https://drive.google.com/uc?id=1YPWqAfXbltU56Biz7j2_nyAWlyODEdUA&confirm=t&uuid=098a00c8-1c48-41af-9d39-aaf3b4c8abc8
To: /home/linn/2024/dec/aiml-handson/galactic-rings/E11dash.zip
100%|██████████████████████████████████████| 67.7M/67.7M [00:05<00:00, 11.5MB/s]


In [56]:
!echo "4925fae1310595ce8fa7f47e98180953  E11dash.zip" | md5sum -c

E11dash.zip: OK


In [58]:
%%capture
!unzip E11dash.zip

In [29]:
import argparse
from tensorflow.keras import layers
from tensorflow.keras.models import load_model
from sklearn.metrics import confusion_matrix
from sklearn.metrics import accuracy_score, precision_score, recall_score, \
f1_score, roc_auc_score, roc_curve, balanced_accuracy_score, brier_score_loss, \
average_precision_score, fbeta_score, matthews_corrcoef, auc, precision_recall_curve, \
classification_report
import numpy as np

In [49]:
%%capture
!pip install scikit-learn

In [7]:
eval_parser = argparse.ArgumentParser()

In [8]:
eval_parser.add_argument('--trained-model', required=True, help="Path to trained model")
eval_parser.add_argument('--test-dir',  help="Directory containing validation or test images sorted into respective classes")
eval_parser.add_argument('--saved-ds',  default=False, help="Boolean flag that is true if test_dir points to a tf.data.Dataset object")
eval_parser.add_argument('--threshold', type=float,  default=0.5, help="Decimal threshold to use for creating CM, etc.")
eval_parser.add_argument('--write', action="store_true", help="Switch to enable writing results to disk")

_StoreTrueAction(option_strings=['--write'], dest='write', nargs=0, const=True, default=False, type=None, choices=None, required=False, help='Switch to enable writing results to disk', metavar=None)

In [35]:
eval_args = eval_parser.parse_args(['--test-dir','E11dash/test/','--trained-model','clean-shadow-84-slim/best_model.h5'])

In [36]:
test_dir = eval_args.test_dir
model_path = eval_args.trained_model
batch_size = 64

In [37]:
img_height, img_width = args.target_size

In [38]:
test_ds = tf.keras.utils.image_dataset_from_directory(
  test_dir,
  #color_mode='grayscale',
  shuffle=False,
  image_size=(img_height, img_width),
  batch_size=None)

filenames = test_ds.file_paths

normalization_layer = layers.Rescaling(1./255)

test_ds = test_ds.map(lambda x, y: (normalization_layer(x), y))

labels = test_ds.map(lambda _, label: label)

AUTOTUNE = tf.data.AUTOTUNE
test_ds = test_ds.batch(batch_size).cache().prefetch(buffer_size=AUTOTUNE)

Found 2353 files belonging to 2 classes.


In [39]:
model = load_model(model_path)

In [40]:
ground_truth = list(labels.as_numpy_iterator())
predictions = model.predict(test_ds)

2025-01-07 09:52:03.889018: I tensorflow/core/common_runtime/executor.cc:1197] [/device:CPU:0] (DEBUG INFO) Executor start aborting (this does not indicate an error and you can ignore this message): INVALID_ARGUMENT: You must feed a value for placeholder tensor 'Placeholder/_4' with dtype int32 and shape [2353]
	 [[{{node Placeholder/_4}}]]
2025-01-07 09:52:03.889574: I tensorflow/core/common_runtime/executor.cc:1197] [/device:CPU:0] (DEBUG INFO) Executor start aborting (this does not indicate an error and you can ignore this message): INVALID_ARGUMENT: You must feed a value for placeholder tensor 'Placeholder/_4' with dtype int32 and shape [2353]
	 [[{{node Placeholder/_4}}]]
2025-01-07 09:52:05.029093: I tensorflow/core/common_runtime/executor.cc:1197] [/device:CPU:0] (DEBUG INFO) Executor start aborting (this does not indicate an error and you can ignore this message): INVALID_ARGUMENT: You must feed a value for placeholder tensor 'Placeholder/_0' with dtype string and shape [2353]


37/37 [==============================] - 2s 31ms/step


In [41]:
threshold = eval_args.threshold
print("Using a classification threshold", threshold)

predicted_labels = [1 if pred >= threshold else 0 for pred in predictions]

Using a classification threshold 0.5


In [42]:
confusion_mtx = confusion_matrix(ground_truth, predicted_labels)
print("Confusion Matrix:")
print(confusion_mtx)

Confusion Matrix:
[[2093   35]
 [  12  213]]


In [43]:
accuracy = accuracy_score(ground_truth, predicted_labels)
f1 = f1_score(ground_truth, predicted_labels)
try:
    roc_auc = roc_auc_score(ground_truth, predictions)
except:
    print("Setting roc_auc to be -1 as it is not defined")
    roc_auc = -1
precisions, recalls, thresholds = precision_recall_curve(ground_truth, predictions)
pr_auc = auc(recalls, precisions)
brier_score = brier_score_loss(ground_truth, predictions)
avg_precision = average_precision_score(ground_truth, predictions)
report = classification_report(ground_truth, predicted_labels, target_names=['NonRings', 'Rings'])

tn, fp, fn, tp = confusion_mtx.ravel()
fpr = fp / (fp + tn)
specificity = tn / (fp + tn)
precision = precision_score(ground_truth, predicted_labels)
recall = recall_score(ground_truth, predicted_labels)
bal_acc = balanced_accuracy_score(ground_truth, predicted_labels)
matthews_corr_coef = matthews_corrcoef(ground_truth, predicted_labels)
beta = 2
fbeta = fbeta_score(ground_truth, predicted_labels, beta=beta)
print("Accuracy:", accuracy)
print("F1-score:", f1)
print("ROC AUC Score:", roc_auc)
print("PR AUC Score:", pr_auc)
print("Brier score", brier_score)
print("Average precision score", avg_precision)
print("Classification Report")
print(report)
print("False Positive Rate (FPR):", fpr)
print("TNR or Specificity:", specificity)
print("G-Mean:", np.sqrt(recall * specificity))
print(f"F-beta score beta={beta}", fbeta)
print("F1 Score After Thresholding: {}".format( f1_score(ground_truth, predicted_labels)))
print("Matthew Correlation Coefficient:", matthews_corr_coef)
print("Balanced Accuracy:", bal_acc)

Accuracy: 0.9800254993625159
F1-score: 0.9006342494714588
ROC AUC Score: 0.9950730994152046
PR AUC Score: 0.922037758773358
Brier score 0.015240331209888786
Average precision score 0.924448016803026
Classification Report
              precision    recall  f1-score   support

    NonRings       0.99      0.98      0.99      2128
       Rings       0.86      0.95      0.90       225

    accuracy                           0.98      2353
   macro avg       0.93      0.97      0.94      2353
weighted avg       0.98      0.98      0.98      2353

False Positive Rate (FPR): 0.01644736842105263
TNR or Specificity: 0.9835526315789473
G-Mean: 0.9649334128467467
F-beta score beta=2 0.9277003484320558
F1 Score After Thresholding: 0.9006342494714588
Matthew Correlation Coefficient: 0.8908621868910627
Balanced Accuracy: 0.9651096491228071
